In [1]:
import rasterio
from pathlib import Path
import pandas as pd
from dist_s1_enumerator import enumerate_one_dist_s1_product
from dist_s1.data_models.data_utils import get_max_pre_imgs_per_burst_mw

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [216]:
products = list(Path('sample_prods').glob('*.tif'))
sample_layer = products[-1]
sample_layer

PosixPath('sample_prods/OPERA_L3_DIST-ALERT-S1_T11SLT_20250619T140054Z_20250911T163102Z_S1_30_v0.1_GEN-DIST-STATUS.tif')

In [191]:
def get_burst_id(opera_rtc_id: str) -> str:
    return opera_rtc_id.split('_')[3]

def get_track_number(burst_id: str) -> int:
    return int(burst_id.split('-')[0][1:])

def get_acq_time(opera_rtc_id: str) -> pd.Timestamp:
    return pd.Timestamp(opera_rtc_id.split('_')[4])

def get_rtc_input_data(layer_path: str | Path) -> dict:
    with rasterio.open(layer_path) as ds:
        tags = ds.tags()

    rtc_inputs = {
        'post_rtc_opera_ids': tags['post_rtc_opera_ids'].split(','),
        'pre_rtc_opera_ids': tags['pre_rtc_opera_ids'].split(','),
        'mgrs_tile_id': tags['mgrs_tile_id'],  
    }
    
    return rtc_inputs

def format_rtc_input_data(rtc_data: dict) -> pd.DataFrame:
    n_pre = len(rtc_data['pre_rtc_opera_ids'])
    df_pre = pd.DataFrame({'opera_id': rtc_data['pre_rtc_opera_ids'],
                           'input_category': ['pre'] * n_pre})
    n_post = len(rtc_data['post_rtc_opera_ids'])
    df_post = pd.DataFrame({'opera_id': rtc_data['post_rtc_opera_ids'],
                           'input_category': ['post'] * n_post})
    df = pd.concat([df_pre, df_post])
    df['opera_id_trunc'] = df.opera_id.map(lambda opera_id: '_'.join(opera_id.split('_')[:5]))

    df['jpl_burst_id'] = df.opera_id.map(get_burst_id)
    df['track_number'] = df.jpl_burst_id.map(get_track_number)
    df['mgrs_tile_id'] = rtc_data['mgrs_tile_id']
    df['acq_dt'] = df.opera_id.map(get_acq_time)
    return df

In [192]:
data = get_rtc_input_data(sample_layer)
df = format_rtc_input_data(data)
df.head()

,opera_id,input_category,opera_id_trunc,jpl_burst_id,track_number,mgrs_tile_id,acq_dt
0,OPERA_L2_RTC-S1_T144-308024-IW1_20240519T14010...,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20240519T140104Z,T144-308024-IW1,144,11SLT,2024-05-19 14:01:04+00:00
1,OPERA_L2_RTC-S1_T144-308024-IW1_20240531T14010...,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20240531T140104Z,T144-308024-IW1,144,11SLT,2024-05-31 14:01:04+00:00
2,OPERA_L2_RTC-S1_T144-308024-IW1_20240612T14010...,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20240612T140103Z,T144-308024-IW1,144,11SLT,2024-06-12 14:01:03+00:00
3,OPERA_L2_RTC-S1_T144-308024-IW1_20220518T14005...,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20220518T140054Z,T144-308024-IW1,144,11SLT,2022-05-18 14:00:54+00:00
4,OPERA_L2_RTC-S1_T144-308024-IW1_20220530T14005...,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20220530T140055Z,T144-308024-IW1,144,11SLT,2022-05-30 14:00:55+00:00


In [193]:
track_numbers = df.track_number.unique().tolist()
if len(track_numbers) > 1:
    if abs(track_numbers[0] - track_numbers[1]) > 1:
        print(f'too many track numbers present: {track_numbers}')

In [194]:
post_ind = df.input_category == 'post'
df_post = df[post_ind].reset_index(drop=True)

pre_ind = df.input_category == 'pre'
df_pre = df[pre_ind].reset_index(drop=True)

In [195]:
post_time_delta = df_post.acq_dt.max() - df_post.acq_dt.min()
if (post_time_delta).days > 1:
    print(f'post-dates span too long: {post_time_delta}')
post_time_delta

Timedelta('0 days 00:00:20')

In [196]:
max_pre_imgs_per_burst = get_max_pre_imgs_per_burst_mw(10, 3)
max_pre_imgs_per_burst

(3, 3, 4)

In [197]:
df_product_expected = enumerate_one_dist_s1_product(
        df.mgrs_tile_id.iloc[0],
        track_number=track_numbers[0],
        post_date=str(df_post.acq_dt.min().date()),
        lookback_strategy='multi_window',
        delta_lookback_days=(1095, 730, 365),
        max_pre_imgs_per_burst=max_pre_imgs_per_burst
    )

Searching for post-images for track 144 in MGRS tile 11SLT
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365


Windows: 100%|█████████████████████████████████████████████████████| 3/3 [00:11<00:00,  3.70s/it]


In [198]:
df_product_expected['opera_id_trunc'] = df_product_expected.opera_id.map(lambda opera_id: '_'.join(opera_id.split('_')[:5]))

df_product_expected.shape, df.shape

((88, 15), (80, 7))

In [199]:
df_product_expected.head()

,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token,input_category,opera_id_trunc
0,OPERA_L2_RTC-S1_T144-308024-IW1_20220518T14005...,T144-308024-IW1,2022-05-18 14:00:54+00:00,2022-05-18,VV+VH,144,509,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-118.33347 34.45184, -119.31496 34.6...",11SLT,3,144,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20220518T140054Z
1,OPERA_L2_RTC-S1_T144-308024-IW1_20220530T14005...,T144-308024-IW1,2022-05-30 14:00:55+00:00,2022-05-30,VV+VH,144,511,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-118.33557 34.45162, -119.31707 34.6...",11SLT,3,144,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20220530T140055Z
2,OPERA_L2_RTC-S1_T144-308024-IW1_20220611T14005...,T144-308024-IW1,2022-06-11 14:00:56+00:00,2022-06-11,VV+VH,144,513,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-118.33622 34.45086, -119.31767 34.6...",11SLT,3,144,pre,OPERA_L2_RTC-S1_T144-308024-IW1_20220611T140056Z
3,OPERA_L2_RTC-S1_T144-308025-IW1_20220518T14005...,T144-308025-IW1,2022-05-18 14:00:57+00:00,2022-05-18,VV+VH,144,509,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-118.36562 34.28459, -119.33426 34.4...",11SLT,3,144,pre,OPERA_L2_RTC-S1_T144-308025-IW1_20220518T140057Z
4,OPERA_L2_RTC-S1_T144-308025-IW1_20220530T14005...,T144-308025-IW1,2022-05-30 14:00:58+00:00,2022-05-30,VV+VH,144,511,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-118.36766 34.28462, -119.33632 34.4...",11SLT,3,144,pre,OPERA_L2_RTC-S1_T144-308025-IW1_20220530T140058Z


In [200]:
post_ind = df_product_expected.input_category == 'post'
df_product_expected_post = df_product_expected[post_ind].reset_index(drop=True)

pre_ind = df_product_expected.input_category == 'pre'
df_product_expected_pre = df_product_expected[pre_ind].reset_index(drop=True)

## Burst ids

In [201]:
burst_id_expected = sorted(df_product_expected.jpl_burst_id.unique().tolist())
bust_id_prod = sorted(df.jpl_burst_id.unique().tolist())

In [202]:
burst_id_expected == bust_id_prod

True

In [203]:
len(burst_id_expected)

8

## Product ids

In [204]:
pre_rtc_ids_expected_but_not_found = [rtc_id for rtc_id in df_product_expected_pre.opera_id_trunc.tolist()
                                              if rtc_id not in df_pre.opera_id_trunc.tolist()]
pre_rtc_ids_expected_but_not_found

['OPERA_L2_RTC-S1_T144-308024-IW1_20240507T140104Z',
 'OPERA_L2_RTC-S1_T144-308025-IW1_20240507T140107Z',
 'OPERA_L2_RTC-S1_T144-308026-IW1_20240507T140110Z',
 'OPERA_L2_RTC-S1_T144-308027-IW1_20240507T140113Z',
 'OPERA_L2_RTC-S1_T144-308028-IW1_20240507T140115Z',
 'OPERA_L2_RTC-S1_T144-308029-IW1_20240507T140118Z',
 'OPERA_L2_RTC-S1_T144-308030-IW1_20240507T140121Z',
 'OPERA_L2_RTC-S1_T144-308031-IW1_20240507T140124Z']

In [205]:
pre_found_but_not_expected = [rtc_id for rtc_id in df_pre.opera_id_trunc.tolist()
                               if rtc_id not in df_product_expected_pre.opera_id_trunc.tolist()]
pre_found_but_not_expected

[]

In [206]:
post_rtc_ids_expected_but_not_found = [rtc_id for rtc_id in df_product_expected_post.opera_id_trunc.tolist()
                                              if rtc_id not in df_post.opera_id_trunc.tolist()]
post_rtc_ids_expected_but_not_found

[]

In [207]:
post_found_but_not_expected = [rtc_id for rtc_id in df_post.opera_id_trunc.tolist()
                               if rtc_id not in df_product_expected_post.opera_id_trunc.tolist()]
post_found_but_not_expected

[]

# Histogram

In [208]:
from typing import Dict, Tuple, Any

def count_acquisitions_by_window(
    df_prod: pd.DataFrame, ref_date: Any
) -> Dict[str, Tuple[int, int, int]]:

    df_prod['days_ago'] = (ref_date - df_prod['acq_dt']).dt.days

    # 3. Define the bins for 'days_ago'
    # Bins: (365, 730], (730, 1095], (1095, 1460]
    bins = [365, 730, 1095, 1500]
    labels = ['W1', 'W2', 'W3'] 

    # Use pd.cut to assign each row to a time window
    df_prod['time_window'] = pd.cut(df_prod['days_ago'], bins=bins, labels=labels, right=False)

    # 4. Group by 'burst_id' and 'time_window', then unstack to get the counts
    counts_df = (
        df_prod.dropna(subset=['time_window'])
        .groupby(['jpl_burst_id', 'time_window'])
        .size()
        .unstack(fill_value=0)
    )

    # 5. Ensure all 3 window columns exist and are in the correct order
    for label in labels:
        if label not in counts_df.columns:
            counts_df[label] = 0

    counts_df = counts_df[labels]

    # 6. Convert the resulting DataFrame to the desired dictionary format
    result_dict = counts_df.apply(tuple, axis=1).to_dict()

    return result_dict


def get_ordered_dates_by_burst(df: pd.DataFrame) -> dict:
    df_copy = df.copy()
    def sort_and_format_dates(series: pd.Series) -> list[str]:
        sorted_dates = series.sort_values(ascending=False)
        return sorted_dates.dt.strftime('%Y-%m-%d').tolist()

    ordered_dates = (
        df_copy.groupby('jpl_burst_id')['acq_dt']
        .apply(sort_and_format_dates)
        .to_dict()
    )

    return ordered_dates

In [209]:
count_acquisitions_by_window(df_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_27366/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T144-308024-IW1': (3, 3, 3),
 'T144-308025-IW1': (3, 3, 3),
 'T144-308026-IW1': (3, 3, 3),
 'T144-308027-IW1': (3, 3, 3),
 'T144-308028-IW1': (3, 3, 3),
 'T144-308029-IW1': (3, 3, 3),
 'T144-308030-IW1': (3, 3, 3),
 'T144-308031-IW1': (3, 3, 3)}

In [210]:
count_acquisitions_by_window(df_product_expected_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_27366/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T144-308024-IW1': (4, 3, 3),
 'T144-308025-IW1': (4, 3, 3),
 'T144-308026-IW1': (4, 3, 3),
 'T144-308027-IW1': (4, 3, 3),
 'T144-308028-IW1': (4, 3, 3),
 'T144-308029-IW1': (4, 3, 3),
 'T144-308030-IW1': (4, 3, 3),
 'T144-308031-IW1': (4, 3, 3)}

In [211]:
pd.Timestamp('2023-12-10') - pd.Timestamp('2021-12-14')

Timedelta('726 days 00:00:00')

In [212]:
print(df_product_expected_post.acq_dt.max())
get_ordered_dates_by_burst(df_product_expected_pre)

2025-06-19 14:01:14+00:00


{'T144-308024-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308025-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308026-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308027-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308028-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308029-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
 

In [213]:
count_acquisitions_by_window(df_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_27366/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T144-308024-IW1': (3, 3, 3),
 'T144-308025-IW1': (3, 3, 3),
 'T144-308026-IW1': (3, 3, 3),
 'T144-308027-IW1': (3, 3, 3),
 'T144-308028-IW1': (3, 3, 3),
 'T144-308029-IW1': (3, 3, 3),
 'T144-308030-IW1': (3, 3, 3),
 'T144-308031-IW1': (3, 3, 3)}

In [214]:
print(df_product_expected_post.acq_dt.max())
get_ordered_dates_by_burst(df_pre)

2025-06-19 14:01:14+00:00


{'T144-308024-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308025-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308026-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308027-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308028-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308029-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308030-IW

In [215]:
print(df_product_expected_post.acq_dt.max())
get_ordered_dates_by_burst(df_product_expected_pre)

2025-06-19 14:01:14+00:00


{'T144-308024-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308025-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308026-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308027-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308028-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
  '2023-06-06',
  '2023-05-25',
  '2022-06-11',
  '2022-05-30',
  '2022-05-18'],
 'T144-308029-IW1': ['2024-06-12',
  '2024-05-31',
  '2024-05-19',
  '2024-05-07',
  '2023-06-18',
 